# BERT Sentiment Classification: Training Trade-offs and Error Analysis

**Joe Williams · Completed Colab pilot · November 15, 2026**

**Question:** Under one fixed configuration, how do classifier-only training and full BERT fine-tuning compare in predictive performance, training time, and errors?

The baseline is adapted from *Hands-On Large Language Models* (Jay Alammar and Maarten Grootendorst, 2024), Chapter 11. Original code copyright 2024 Jay Alammar & Maarten Grootendorst; Apache License 2.0. [Chapter source](https://github.com/HandsOnLLM/Hands-On-Large-Language-Models/tree/main/chapter11). See `ATTRIBUTION.md` and the retained `LICENSE` for the sources and changes.

**Main result:** full fine-tuning reached 85.46% validation accuracy versus 60.69% for classifier-only training, taking 85.02 versus 36.24 seconds for the training call. The selected full model scored 85.08% on the official test split. This is one seed and one epoch, without regime-specific tuning.

This notebook preserves the completed experiment's training/evaluation logic and saved numerical outputs. One unsupported metadata field was removed from the final-result construction after execution; that edit is documented below. Explanatory Markdown was added after execution. Interactive progress/download displays and the displayed raw review excerpt were removed. No training was performed during packaging.

**Read the saved outputs without running all cells.** The preserved execution cells have training and final testing enabled. For a new run, use a separate copy, first disable those switches in their execution cells, and pin the two revisions from the data record. Keep the original run separate.

## 1. Dependencies

These compatibility ranges are the original installation settings. They are not a lockfile; actual installed versions appear below.

In [3]:
# Installing the project's dependencies without forcing an upgrade of Colab's existing PyTorch.
%pip install -q "transformers>=4.46,<6" "datasets>=3,<5" "accelerate>=1,<2" "huggingface-hub>=0.26,<2" "numpy>=1.26,<3" "pandas>=2.2,<4" "scikit-learn>=1.5,<2"

## 2. Environment and run directory

The run identifier ties the outputs together. The recorded runtime used a Tesla T4. A new execution creates a new artifact directory.

In [4]:
# Import the standard-library tools used for artifact records.
import gc, hashlib, importlib.metadata, json, platform, time, uuid
# Import timestamp helpers.
from datetime import datetime, timezone
# Import path handling.
from pathlib import Path
# Import numerical operations.
import numpy as np
# Import tables for reviewing results.
import pandas as pd
# Import PyTorch for model execution and hardware information.
import torch
# Import the notebook display function.
from IPython.display import display
# Import the dataset loader.
from datasets import load_dataset
# Import the Hub metadata API for recording immutable revisions.
from huggingface_hub import HfApi
# Import the classifier and current training API.
from transformers import AutoTokenizer, AutoModelForSequenceClassification, DataCollatorWithPadding, TrainingArguments, Trainer, set_seed
# Choose a fresh run identifier so artifacts are not silently overwritten.
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "-" + uuid.uuid4().hex[:6]
# Define this run's output directory.
ARTIFACTS = Path("artifacts") / RUN_ID
# Create it without modifying earlier runs.
ARTIFACTS.mkdir(parents=True, exist_ok=False)
# Record the installed versions relevant to reproduction.
PACKAGES = ["torch", "transformers", "datasets", "accelerate", "huggingface-hub", "numpy", "pandas", "scikit-learn"]
# Store environment information for this run.
environment = {"python": platform.python_version(), "packages": {name: importlib.metadata.version(name) for name in PACKAGES}, "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None, "cuda": torch.version.cuda}
# Save the environment record.
(ARTIFACTS / "environment.json").write_text(json.dumps(environment, indent=2))
# Display the record for your review.
print(json.dumps(environment, indent=2))

{
  "python": "3.13.16",
  "packages": {
    "torch": "2.14.1+cu130",
    "transformers": "5.22.1",
    "datasets": "4.8.5",
    "accelerate": "1.15.0",
    "huggingface-hub": "1.33.0",
    "numpy": "2.4.6",
    "pandas": "2.2.3",
    "scikit-learn": "1.6.1"
  },
  "gpu": "Tesla T4",
  "cuda": "13.0"
}


## 3. Fixed experimental settings

Both regimes use seed 42, one epoch, batch size 16, learning rate 0.00002, and maximum length 256. The score threshold of 0.90 is descriptive. Softmax scores are not calibrated probabilities of correctness.

In [5]:
# Set the public BERT repository.
MODEL_ID = "google-bert/bert-base-cased"
# Set the public dataset repository.
DATASET_ID = "cornell-movie-review-data/rotten_tomatoes"
# Fix the seed for this pilot, while recognizing hardware can still affect reproducibility.
SEED = 42
# Limit the input sequence length consistently.
MAX_LENGTH = 256
# Keep the classifier training comparison to one epoch.
EPOCHS = 1
# Set the same batch size for both regimes.
BATCH_SIZE = 16
# Set the same learning rate for both regimes.
LEARNING_RATE = 2e-5
# Declare the descriptive confidence threshold in advance.
CONFIDENCE_THRESHOLD = 0.90
# Declare the two regimes and their run order; there is no bonus sweep.
REGIMES = ["head_only", "full_finetune"]
# Keep expensive training disabled until you intentionally enable it in section 6.
RUN_TRAINING = False
# Keep final test evaluation disabled until validation review is complete.
RUN_FINAL_TEST = False
# Require a GPU by default to avoid an unexpectedly slow CPU training run.
ALLOW_CPU_TRAINING = False
# Record the settings before obtaining results.
experiment_config = {"model_id": MODEL_ID, "dataset_id": DATASET_ID, "seed": SEED, "max_length": MAX_LENGTH, "epochs": EPOCHS, "batch_size": BATCH_SIZE, "learning_rate": LEARNING_RATE, "regimes": REGIMES, "confidence_threshold": CONFIDENCE_THRESHOLD}
# Save the configuration for the run.
(ARTIFACTS / "experiment_config.json").write_text(json.dumps(experiment_config, indent=2))
# Display the planned comparison.
print(json.dumps(experiment_config, indent=2))

{
  "model_id": "google-bert/bert-base-cased",
  "dataset_id": "cornell-movie-review-data/rotten_tomatoes",
  "seed": 42,
  "max_length": 256,
  "epochs": 1,
  "batch_size": 16,
  "learning_rate": 2e-05,
  "regimes": [
    "head_only",
    "full_finetune"
  ],
  "confidence_threshold": 0.9
}


## 4. Data and revisions

The official train, validation, and test splits contain 8,530 / 1,066 / 1,066 examples. Labels are 0 = negative and 1 = positive. The exact-text check normalizes case and whitespace; zero overlaps do not rule out semantic duplicates.

The two source lines query the latest Hub revisions for a new run. To reuse this run's snapshots, assign the recorded revision strings from `data_record.json` instead.

In [6]:
# Query the revision of the public model used in this run.
model_revision = HfApi().model_info(MODEL_ID).sha
# Query the revision of the public dataset used in this run.
dataset_revision = HfApi().dataset_info(DATASET_ID).sha
# Load that exact dataset snapshot.
dataset = load_dataset(DATASET_ID, revision=dataset_revision)
# Load the tokenizer from the matching model snapshot.
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, revision=model_revision)
# Verify that all expected splits exist.
assert {"train", "validation", "test"}.issubset(dataset.keys()), "Expected train, validation, and test splits."
# Read the dataset's own class labels rather than assuming their order silently.
label_names = dataset["train"].features["label"].names
# Confirm the expected polarity mapping used by this experiment.
assert label_names == ["neg", "pos"], f"Review the dataset's label mapping: {label_names}"
# Define tokenization once for every split.
def tokenize_batch(batch):
    # Apply identical preprocessing to both training regimes.
    return tokenizer(batch["text"], truncation=True, max_length=MAX_LENGTH)
# Tokenize all splits without retaining raw text in the training inputs.
tokenized = dataset.map(tokenize_batch, batched=True, remove_columns=["text"])
# Prepare padding at the batch level.
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
# Define a conservative exact-text overlap audit, not a semantic-duplicate detector.
text_sets = {split: {" ".join(text.lower().split()) for text in dataset[split]["text"]} for split in ["train", "validation", "test"]}
# Record overlap counts without writing review texts to disk.
overlap = {"train_validation_exact": len(text_sets["train"] & text_sets["validation"]), "train_test_exact": len(text_sets["train"] & text_sets["test"]), "validation_test_exact": len(text_sets["validation"] & text_sets["test"])}
# Record dataset fingerprints, class labels, and revisions for this run.
data_record = {"model_revision": model_revision, "dataset_revision": dataset_revision, "label_names": label_names, "rows": {split: len(dataset[split]) for split in dataset}, "fingerprints": {split: dataset[split]._fingerprint for split in dataset}, "exact_text_overlap": overlap}
# Save the record without embedding dataset text.
(ARTIFACTS / "data_record.json").write_text(json.dumps(data_record, indent=2))
# Display the audit and counts for review.
print(json.dumps(data_record, indent=2))

{
  "model_revision": "cd5ef92a9fb2f889e972770a36d4ed042daf221e",
  "dataset_revision": "aa13bc287fa6fcab6daf52f0dfb9994269ffea28",
  "label_names": [
    "neg",
    "pos"
  ],
  "rows": {
    "train": 8530,
    "validation": 1066,
    "test": 1066
  },
  "fingerprints": {
    "train": "0346a9d2308d9056",
    "validation": "4d310508c191379e",
    "test": "23aa2ff3ad95b6f5"
  },
  "exact_text_overlap": {
    "train_validation_exact": 0,
    "train_test_exact": 0,
    "validation_test_exact": 0
  }
}


## 5. Metrics and trainable parameters

Metrics expose their denominators. If no predictions meet the score threshold, the conditional error rate is undefined (`null` in JSON), not zero. The freezing helper disables gradients for the whole BERT backbone in the classifier-only regime.

In [7]:
# Metric and parameter-configuration helpers; see ATTRIBUTION.md.
# These functions do not train a model or contact an external service.
import numpy as np
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score


def prediction_details(logits, labels):
    """Validate binary predictions and return predictions, confidence, and correctness."""
    # Convert logits to a stable numerical representation.
    scores = np.asarray(logits, dtype=np.float64)
    # Preserve labels until they have been checked.
    actual = np.asarray(labels)
    # Require one row per example and exactly two class scores.
    if scores.ndim != 2 or scores.shape[1] != 2 or scores.shape[0] == 0:
        raise ValueError("Expected a nonempty (number_of_examples, 2) logits array.")
    # Reject invalid scores instead of producing misleading metrics.
    if not np.isfinite(scores).all():
        raise ValueError("Logits must contain only finite values.")
    # Require a matching label vector with labels 0 and 1.
    if actual.ndim != 1 or len(actual) != len(scores) or not np.isin(actual, [0, 1]).all():
        raise ValueError("Labels must be a matching one-dimensional array of 0s and 1s.")
    # Convert validated labels to integers.
    actual = actual.astype(np.int64)
    # Subtract each row's maximum to prevent exponential overflow.
    shifted = scores - scores.max(axis=1, keepdims=True)
    # Convert shifted logits to positive weights.
    weights = np.exp(shifted)
    # Normalize each row to obtain softmax scores.
    probabilities = weights / weights.sum(axis=1, keepdims=True)
    # Pick the highest-scoring class; ties select class 0.
    predicted = probabilities.argmax(axis=1)
    # Store the selected class's score, not a calibrated probability of correctness.
    confidence = probabilities.max(axis=1)
    # Record whether each predicted label matches the dataset label.
    correct = predicted == actual
    # Return arrays needed for both metrics and error review.
    return actual, predicted, confidence, correct


def classification_summary(logits, labels, confidence_threshold=0.90):
    """Return binary/macro F1, accuracy, confusion counts, and confident-error counts."""
    # Validate the descriptive threshold before using it.
    if not 0.5 <= confidence_threshold <= 1.0:
        raise ValueError("The confidence threshold must be between 0.5 and 1.0.")
    # Convert predictions and validate the inputs in one place.
    actual, predicted, confidence, correct = prediction_details(logits, labels)
    # Identify predictions above the declared threshold.
    confident = confidence >= confidence_threshold
    # Count the denominator explicitly.
    confident_count = int(confident.sum())
    # Count mistakes within that selected group.
    confident_errors = int((confident & ~correct).sum())
    # Use null rather than zero when the conditional rate has no denominator.
    confident_error_rate = confident_errors / confident_count if confident_count else None
    # Return JSON-compatible values with unambiguous metric names.
    return {
        "n_examples": int(len(actual)),
        "accuracy": float(accuracy_score(actual, predicted)),
        "binary_f1_positive": float(f1_score(actual, predicted, pos_label=1, zero_division=0)),
        "macro_f1": float(f1_score(actual, predicted, labels=[0, 1], average="macro", zero_division=0)),
        "confusion_matrix_rows_actual_columns_predicted": confusion_matrix(actual, predicted, labels=[0, 1]).tolist(),
        "confidence_threshold": float(confidence_threshold),
        "confident_count": confident_count,
        "confident_errors": confident_errors,
        "confident_error_rate": confident_error_rate,
    }


def configure_trainable_parameters(model, regime):
    """Configure a BERT sequence classifier without relying on parameter positions."""
    # Reject spelling mistakes or unsupported experiments.
    if regime not in {"head_only", "full_finetune"}:
        raise ValueError("regime must be 'head_only' or 'full_finetune'.")
    # Require the architecture this experiment was designed for.
    if not hasattr(model, "bert") or not hasattr(model, "classifier"):
        raise TypeError("This experiment expects a BERT model with a classifier head.")
    # Reset all parameters so earlier calls cannot leak a freezing configuration.
    for parameter in model.parameters():
        parameter.requires_grad = True
    # Freeze the complete BERT backbone for the classifier-only experiment.
    if regime == "head_only":
        for parameter in model.bert.parameters():
            parameter.requires_grad = False
    # Count trainable scalar parameters rather than parameter tensors.
    trainable = sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)
    # Count every scalar parameter for a meaningful denominator.
    total = sum(parameter.numel() for parameter in model.parameters())
    # Ensure the experiment can actually update something.
    if trainable == 0:
        raise ValueError("No trainable parameters remain.")
    # Return counts for the run record.
    return {"trainable_parameters": int(trainable), "total_parameters": int(total)}

## 6. Train and compare

Each regime starts from the same pretrained checkpoint and resets the seed before creating the classifier. The recorded initialization hash covers the classifier weight tensor. Full BERT dropout remains active during classifier-only training; this is not cached-feature training.

Training-call wall time excludes model loading, validation, and saving. Each regime saves validation predictions and metrics before the next regime starts.

In [8]:
# Define the small metric callback used by the Transformers Trainer.
def compute_metrics(eval_prediction):
    # Calculate metrics from real predictions returned by this run.
    summary = classification_summary(eval_prediction.predictions, eval_prediction.label_ids, CONFIDENCE_THRESHOLD)
    # Return only scalar metrics to the Trainer's logger.
    return {key: summary[key] for key in ["accuracy", "binary_f1_positive", "macro_f1"]}

# Define an isolated run so variables from SetFit, MLM, or NER cannot leak into it.
def run_regime(regime):
    # Reset the random seed before the randomly initialized classification head is created.
    set_seed(SEED)
    # Load the same public checkpoint for each regime, using safetensors weights.
    candidate = AutoModelForSequenceClassification.from_pretrained(MODEL_ID, revision=model_revision, num_labels=2, id2label={0: "NEGATIVE", 1: "POSITIVE"}, label2id={"NEGATIVE": 0, "POSITIVE": 1}, use_safetensors=True)
    # Record the initial head as a check that initialization was identical across regimes.
    head_hash = hashlib.sha256(candidate.classifier.weight.detach().cpu().numpy().tobytes()).hexdigest()
    # Configure and count trainable parameters for this regime.
    parameter_counts = configure_trainable_parameters(candidate, regime)
    # Give the model an isolated local checkpoint directory.
    checkpoint = Path("scratch") / RUN_ID / regime
    # Use fixed settings, no intermediate checkpoint saving, and no external reporting.
    arguments = TrainingArguments(output_dir=str(checkpoint), learning_rate=LEARNING_RATE, per_device_train_batch_size=BATCH_SIZE, per_device_eval_batch_size=BATCH_SIZE, num_train_epochs=EPOCHS, weight_decay=0.01, eval_strategy="no", save_strategy="no", report_to="none", push_to_hub=False, seed=SEED, data_seed=SEED, logging_steps=100, dataloader_num_workers=0)
    # Build the Trainer with the validation split, never the test split, for this comparison.
    candidate_trainer = Trainer(model=candidate, args=arguments, train_dataset=tokenized["train"], eval_dataset=tokenized["validation"], processing_class=tokenizer, data_collator=data_collator, compute_metrics=compute_metrics)
    # Start a wall-clock measurement of the training call only.
    started = time.perf_counter()
    # Perform the one-epoch training run.
    train_output = candidate_trainer.train()
    # Measure elapsed training time before validation and model saving.
    elapsed = time.perf_counter() - started
    # Obtain actual validation predictions for analysis.
    output = candidate_trainer.predict(tokenized["validation"], metric_key_prefix="validation")
    # Calculate explicit metrics and denominators from those predictions.
    metrics = classification_summary(output.predictions, output.label_ids, CONFIDENCE_THRESHOLD)
    # Save the checkpoint locally for the later selected-model test.
    candidate_trainer.save_model(str(checkpoint))
    # Save the tokenizer with the model for reloadability.
    tokenizer.save_pretrained(str(checkpoint))
    # Save a small prediction array locally without review texts.
    np.savez_compressed(ARTIFACTS / f"{regime}_validation_predictions.npz", logits=output.predictions, labels=output.label_ids)
    # Assemble a traceable result record for this run.
    record = {"regime": regime, "split": "validation", "seed": SEED, "initial_head_sha256": head_hash, "train_wall_seconds": elapsed, "train_loss": float(train_output.training_loss), "checkpoint": str(checkpoint), **parameter_counts, **metrics}
    # Write this regime's results immediately so they survive a later regime failure.
    (ARTIFACTS / f"{regime}_validation_metrics.json").write_text(json.dumps(record, indent=2))
    # Release the Trainer and model before loading another model.
    del candidate_trainer, candidate, output
    # Request Python garbage collection.
    gc.collect()
    # Release unused cached GPU allocations where applicable.
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    # Return the record for the comparison table.
    return record

### Recorded validation run

The following cell was executed with `RUN_TRAINING=True`. Its table contains the measured results. Do not run it again merely to view those results.

In [14]:
# Change this single value to True only when you are ready to train.
RUN_TRAINING = True
# Avoid starting expensive work just because someone clicked Run all.
if not RUN_TRAINING:
    print("Training not started. Review the experiment, select a GPU, then set RUN_TRAINING = True.")
else:
    # Require explicit permission for CPU training.
    if not torch.cuda.is_available() and not ALLOW_CPU_TRAINING:
        raise RuntimeError("Select a GPU runtime, or deliberately allow CPU training in section 3.")
    # Prevent accidentally overwriting an existing regime's evidence.
    if any((ARTIFACTS / f"{regime}_validation_metrics.json").exists() for regime in REGIMES):
        raise FileExistsError("This run already has results. Start a fresh run before retraining.")
    # Train and evaluate each declared regime.
    validation_records = [run_regime(regime) for regime in REGIMES]
    # Confirm that both random classification heads had the same starting weights.
    assert len({record["initial_head_sha256"] for record in validation_records}) == 1, "Initial heads differed; investigate before comparing."
    # Write the full validation record.
    (ARTIFACTS / "validation_comparison.json").write_text(json.dumps(validation_records, indent=2))
    # Display the measured comparison, not textbook example numbers.
    display(pd.DataFrame(validation_records)[["regime", "macro_f1", "binary_f1_positive", "accuracy", "train_wall_seconds", "trainable_parameters", "confident_count", "confident_errors", "confident_error_rate"]])

[transformers] BertForSequenceClassification LOAD REPORT from: google-bert/bert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


[transformers] BertForSequenceClassification LOAD REPORT from: google-bert/bert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


,regime,macro_f1,binary_f1_positive,accuracy,train_wall_seconds,trainable_parameters,confident_count,confident_errors,confident_error_rate
0,head_only,0.605815,0.626892,0.606942,36.235298,1538,0,0,NaN
1,full_finetune,0.854440,0.859219,0.854597,85.019049,108311810,718,45,0.062674


## 7. Inspect validation mistakes

The displayed rows are the ten highest-scoring errors for full fine-tuning, not a random sample of all predictions. All `correct` values are therefore false by construction. There are 155 full-model validation mistakes in the complete worksheet. The original `my_observation` entries were left blank.

In [15]:
# Select which regime's validation mistakes to inspect first.
REVIEW_REGIME = "full_finetune"
# Point to its real prediction artifact.
prediction_file = ARTIFACTS / f"{REVIEW_REGIME}_validation_predictions.npz"
# Avoid displaying pretend results before training has finished.
if not prediction_file.exists():
    print("Run section 6 before reviewing prediction errors.")
else:
    # Load only the numerical predictions generated by this run.
    with np.load(prediction_file) as saved:
        actual, predicted, confidence, correct = prediction_details(saved["logits"], saved["labels"])
    # Create an error-analysis table using stable split row IDs.
    errors = pd.DataFrame({"validation_row_id": np.arange(len(actual)), "actual_label": actual, "predicted_label": predicted, "softmax_score": confidence, "correct": correct})
    # Rank wrong predictions by score, keeping the original row identifiers.
    error_candidates = errors.loc[~errors["correct"]].sort_values("softmax_score", ascending=False).copy()
    # Leave the interpretation to the person inspecting the example.
    error_candidates["my_observation"] = ""
    # Save a text-free worksheet for your own observations.
    error_candidates.to_csv(ARTIFACTS / f"{REVIEW_REGIME}_error_review.csv", index=False)
    # Display the first ten mistakes without redistributing review text.
    display(error_candidates.head(10))

,validation_row_id,actual_label,predicted_label,softmax_score,correct,my_observation
748,748,0,1,0.986983,False,
705,705,0,1,0.982168,False,
632,632,0,1,0.978362,False,
1050,1050,0,1,0.974012,False,
541,541,0,1,0.971926,False,
890,890,0,1,0.963973,False,
198,198,1,0,0.963047,False,
1042,1042,0,1,0.960827,False,
1001,1001,0,1,0.956721,False,
348,348,1,0,0.955153,False,


### One example: validation row 748

The dataset label is negative (0); the model predicted positive (1) with softmax score 0.986983. The reviewer credits careful production but criticizes the film as mechanical and excessively dutiful. Mixed evaluative wording and indirect criticism are a plausible explanation for the error, not a demonstrated causal account of the model.

This qualitative interpretation is recorded in `analysis/error_analysis.csv`. The original raw-text output was removed from this public-facing copy. Re-running the preserved cell prints the review again.

In [16]:
# Leave disabled until you choose a specific validation row to inspect.
SHOW_REVIEW_TEXT = True
# Replace the row ID with one from your error table.
REVIEW_ROW_ID = 748
# Display text only after deliberate selection.
if SHOW_REVIEW_TEXT:
    # Reject row IDs outside this split.
    if not 0 <= REVIEW_ROW_ID < len(dataset["validation"]):
        raise IndexError("Choose a valid validation row ID.")
    # Print this one example for private manual review.
    print(dataset["validation"][REVIEW_ROW_ID])

## 8. Selected-model test

The recorded rule chooses the highest validation macro F1, breaking ties with shorter training time. It selected full fine-tuning. The test output below is from the supplied run. A post-run metadata cleanup removed one unsupported provenance field from the final-result JSON expression and its saved display. All numeric values and metric computations are unchanged. The execution count describes the original run; the cell was not re-executed after this metadata-only edit.

Final test logits were not exported. The supplied summary can be checked arithmetically, but its metrics cannot be independently recalculated from per-example test predictions in this package.

In [17]:
# Enable the final test deliberately after validation review.
RUN_FINAL_TEST = True
# Keep the test opt-in.
if not RUN_FINAL_TEST:
    print("Final test not started. Finish validation review first.")
else:
    # Require a complete validation comparison.
    comparison_file = ARTIFACTS / "validation_comparison.json"
    # Reject an incomplete experiment.
    if not comparison_file.exists():
        raise FileNotFoundError("Complete section 6 before the final test.")
    # Prevent repeated test evaluation within one run record.
    if (ARTIFACTS / "final_test_metrics.json").exists():
        raise FileExistsError("Final test already recorded for this run.")
    # Load the actual measured validation results.
    records = json.loads(comparison_file.read_text())
    # Select the model using the declared rule, not test performance.
    selected = sorted(records, key=lambda item: (-item["macro_f1"], item["train_wall_seconds"]))[0]
    # Record the selection before reading test predictions.
    (ARTIFACTS / "selection.json").write_text(json.dumps({"rule": "highest_validation_macro_f1_then_fastest_training", "selected_regime": selected["regime"]}, indent=2))
    # Reload the selected locally saved model.
    final_model = AutoModelForSequenceClassification.from_pretrained(selected["checkpoint"], local_files_only=True)
    # Configure evaluation only, with no reporting or upload.
    final_args = TrainingArguments(output_dir=str(Path("scratch") / RUN_ID / "test"), per_device_eval_batch_size=BATCH_SIZE, report_to="none", push_to_hub=False)
    # Build an isolated evaluator.
    final_trainer = Trainer(model=final_model, args=final_args, processing_class=tokenizer, data_collator=data_collator)
    # Obtain test predictions once for this locked choice.
    final_output = final_trainer.predict(tokenized["test"], metric_key_prefix="test")
    # Measure the test performance from those predictions.
    final_metrics = {"regime": selected["regime"], "split": "test", **classification_summary(final_output.predictions, final_output.label_ids, CONFIDENCE_THRESHOLD)}
    # Preserve the test result without inventing or substituting scores.
    (ARTIFACTS / "final_test_metrics.json").write_text(json.dumps(final_metrics, indent=2))
    # Display the complete result with denominators and confusion counts.
    print(json.dumps(final_metrics, indent=2))
    # Release the evaluation model.
    del final_trainer, final_model, final_output
    # Reclaim unused Python objects.
    gc.collect()
    # Reclaim unused cached GPU memory.
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

{
  "regime": "full_finetune",
  "split": "test",
  "n_examples": 1066,
  "accuracy": 0.850844277673546,
  "binary_f1_positive": 0.8526413345690455,
  "macro_f1": 0.8508220917859473,
  "confusion_matrix_rows_actual_columns_predicted": [
    [
      447,
      86
    ],
    [
      73,
      460
    ]
  ],
  "confidence_threshold": 0.9,
  "confident_count": 708,
  "confident_errors": 48,
  "confident_error_rate": 0.06779661016949153
}


## 9. Export run artifacts

This cell packaged the small JSON, NPZ, and CSV artifacts. It excludes local model checkpoints. The original browser-download display has been removed from this notebook copy. The repository contains the exported results under `results/20261115T224719Z-739463/`.

In [18]:
# Import archive creation.
import shutil
# Write an archive of this run's small artifacts, excluding scratch model checkpoints.
archive_path = shutil.make_archive(f"bert_sentiment_{RUN_ID}", "zip", root_dir=ARTIFACTS)
# Display the local archive filename.
print("Artifact archive:", archive_path)
# Download through Colab when available.
try:
    from google.colab import files
    files.download(archive_path)
# Keep non-Colab environments usable.
except ImportError:
    print("Outside Colab: copy the archive from the current working directory.")

Artifact archive: /content/bert_sentiment_20261115T224719Z-739463.zip


## 10. Interpretation and limits

| Measure | Classifier only | Full fine-tuning |
| --- | ---: | ---: |
| Validation accuracy | 60.69% | 85.46% |
| Validation macro F1 | 0.6058 | 0.8544 |
| Training-call wall time | 36.24 s | 85.02 s |
| Trainable parameters | 1,538 | 108,311,810 |
| Validation errors at score ≥ 0.90 | No qualifying predictions | 45 / 718 (6.27%) |

Full fine-tuning gained 24.77 percentage points in validation accuracy at 2.35 times the observed training time. Its official-test accuracy was 85.08% (907 / 1,066), with macro F1 0.8508. High-scoring errors persisted: 48 of 708 test predictions scoring at least 0.90 were wrong (6.78%).

This supports a limited conclusion about these two configurations. A one-epoch, one-seed pilot with a shared learning rate does not show the best achievable performance of either regime. Training times are single measurements. The test split was held out from training and model selection within this workflow. No calibration study, multi-seed study, cross-domain evaluation, or independent qualitative coding was performed.

All saved validation metrics match calculations from the included prediction arrays. The selection rule is consistent with those records. The final summary is internally consistent with its confusion matrix and denominators. See `analysis/verification.json` and run `python scripts/verify_results.py` to repeat the numerical audit without training.